# U.S. County-Level Housing & Demographics — Missing Data Analysis
**Author:** _Your Name Here_  
**Course:** _Your Course_  
**Due:** _As specified on Moodle_

This notebook follows the assignment instructions:
- Submit **both** this `.ipynb` and a **PDF** export of the notebook.
- Keep answers concise, analytical, and evidence-based.
- Use Markdown cells for written responses; keep code clean and commented.

> **How to use this file:**  
> 1) Set `DATA_URL` (or local path) to the dataset and `DICT_URL` (optional) to the data dictionary.  
> 2) Run cells in order.  
> 3) Fill in the written answers where prompted.  
> 4) Export as PDF (Colab: `File → Print` or `File → Download → Download as pdf`).

---

In [ ]:
# === Environment Setup ===
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 150)
print("Pandas:", pd.__version__)

## Data Access
Set the `DATA_URL` to the CSV/Parquet file for **U.S. County-Level Housing & Demographics**.  
If you have a data dictionary, set `DICT_URL` (optional).

In [ ]:
# === Configure data locations ===
DATA_URL = ""  # <-- paste dataset URL or local path here
DICT_URL = ""  # <-- paste data dictionary URL or local path here (optional)

# === Load data ===
if DATA_URL.endswith(".parquet"):
    df = pd.read_parquet(DATA_URL)
else:
    df = pd.read_csv(DATA_URL) if DATA_URL else pd.DataFrame()

print("Rows, Columns:", df.shape)
df.head(3)

# Part I: Understanding the Data (20 points)
_No coding required in this section._

### Q1. (10 pts)
**a) What is this dataset about? Describe the subject in your own words.**  
> *Draft (edit as needed):*  
> This dataset aggregates **county-level** indicators across the U.S. covering **housing** (e.g., home values, rents, vacancy), and **demographics** (e.g., population, age, income, education). Each row represents a county (or county-equivalent), allowing us to compare local housing dynamics alongside socioeconomic context.

**b) What kinds of decisions could be made using this dataset?**  
> *Examples (edit/add):*  
> • Public policy & planning (zoning, affordable housing targets, infrastructure siting)  
> • Investment & lending (market selection, risk scoring, pricing)  
> • Real estate operations (site selection, demand forecasting, rent setting)  
> • Community development (identifying underserved areas, grant prioritization)

**c) Which variables strike you as most important for understanding housing dynamics? Why?**  
> *Examples (tailor to your columns):*  
> • Median home value / median rent → core price signals and affordability baselines  
> • Household income / poverty rate → purchasing power, affordability pressure  
> • Vacancy rate / housing stock growth → supply tightness vs slack  
> • Population & job growth → demand (migration, labor markets)  
> • Age & household size → unit-type demand (e.g., single-family vs multifamily)  
> • Education attainment → proxy for earnings growth & neighborhood change

---

### Q2. (10 pts)
Identify **three** specific, analysis-ready questions this dataset can answer. For each, list variables and business value.

> **Q2-1 Example:** *Which counties are most affordability-constrained?*  
> **Variables:** median_home_value, median_income (or rent/income ratio), poverty_rate.  
> **Value:** Helps policy makers target assistance; lenders and developers assess demand elasticity and default risk.

> **Q2-2 Example:** *Where are housing shortages most acute?*  
> **Variables:** vacancy_rate, housing_unit_growth (past 5–10 yrs), population_growth.  
> **Value:** Guides site selection, permits prioritization, and capital allocation toward supply-constrained markets.

> **Q2-3 Example:** *Which counties show leading indicators of rapid price growth?*  
> **Variables:** population_net_migration, job_growth, rent_growth, inventory_days_on_market, new_permits.  
> **Value:** Early identification of expansion markets for investors, builders, and public infrastructure planning.

# Part II: Missing Data (80 points)
> Run the setup and data loading cells first. Replace variable names to match your dataset.

## Q3a. (10 pts) Quantify and describe the missing data
- Missing counts and percentages by column
- Percent of rows with any missing value

In [ ]:
if df.empty:
    print("⚠️ Please set DATA_URL and reload the dataset above.")
else:
    missing_counts = df.isna().sum()
    missing_pct = (df.isna().mean() * 100).round(2)
    miss_tbl = (pd.DataFrame({'missing_count': missing_counts, 'missing_pct': missing_pct})
                .sort_values('missing_pct', ascending=False))
    display(miss_tbl)
    any_missing_rows_pct = round(df.isna().any(axis=1).mean() * 100, 2)
    print(f"Percentage of rows with ANY missing value: {any_missing_rows_pct}%")

## Q3b. (10 pts) Unique counties & locations
- How many unique counties? List names.
- Identify states/regions if available.

In [ ]:
COUNTY_COL = 'county'  # e.g., 'county_name'
STATE_COL  = 'state'   # e.g., 'state_name'

if df.empty:
    print("⚠️ Please set DATA_URL and reload the dataset above.")
else:
    if COUNTY_COL not in df.columns:
        print(f"⚠️ Column '{COUNTY_COL}' not found. Update COUNTY_COL.")
    else:
        unique_counties = df[COUNTY_COL].dropna().unique().tolist()
        print("Unique counties count:", len(unique_counties))
        print("Counties:", unique_counties)

    if STATE_COL in df.columns and COUNTY_COL in df.columns:
        state_map = df[[COUNTY_COL, STATE_COL]].dropna().drop_duplicates().sort_values([STATE_COL, COUNTY_COL])
        display(state_map.head(30))
        print(f"Unique states/regions: {df[STATE_COL].nunique()}")

## Q3c. (10 pts) Outliers
Identify abnormal values and how you'd validate and handle them (IQR-based example below).

In [ ]:
TARGET_COLS = ['median_home_value', 'median_rent', 'vacancy_rate']  # edit to match your dataset

def iqr_outliers(series, k=1.5):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - k*iqr, q3 + k*iqr
    return (series < lower) | (series > upper), lower, upper

if df.empty:
    print("⚠️ Please set DATA_URL and reload the dataset above.")
else:
    for col in TARGET_COLS:
        if col in df.columns:
            s = pd.to_numeric(df[col], errors='coerce').dropna()
            mask, lo, hi = iqr_outliers(s)
            outliers = s[mask]
            print(f"Column: {col} | Outliers: {mask.sum()} | Bounds: [{lo:.3f}, {hi:.3f}]")
            display(df.loc[outliers.index, [col]].head(10))

## Q4. (50 pts) Missing-value strategy & cleaning
1) Identify variables with missing data.  
2) Decide drop vs keep; justify imputation if keeping.  
3) Choose **any three** variables and implement cleaning.  
4) Explain steps briefly.

In [ ]:
if df.empty:
    print("⚠️ Please set DATA_URL and reload the dataset above.")
else:
    miss_pct = df.isna().mean().sort_values(ascending=False)
    miss_cols = miss_pct[miss_pct > 0].index.tolist()
    display(pd.DataFrame({'missing_pct': (df[miss_cols].isna().mean() * 100).round(2)}))

    IMPUTE_PLAN = {
        'median_home_value': 'group_median:state',
        'median_rent': 'median',
        'vacancy_rate': 'mean',
        'population': 'group_median:state',
        'income_per_capita': 'median',
        'poverty_rate': 'median',
        'education_bachelor_rate': 'group_median:state',
    }

    to_clean = [c for c in IMPUTE_PLAN.keys() if c in df.columns][:3]
    print("Cleaning these variables:", to_clean)

    clean_df = df.copy()
    for col in to_clean:
        strat = IMPUTE_PLAN[col]
        if strat == 'drop_rows':
            before = len(clean_df)
            clean_df = clean_df[~clean_df[col].isna()].copy()
            after = len(clean_df)
            print(f"[{col}] drop_rows: {before - after} rows dropped.")
        elif strat == 'median':
            x = pd.to_numeric(clean_df[col], errors='coerce')
            med = x.median()
            impute_mask = x.isna()
            clean_df.loc[impute_mask, col] = med
            clean_df[col + "_imputed_flag"] = impute_mask.astype(int)
            print(f"[{col}] median impute with {med:.3f}")
        elif strat == 'mean':
            x = pd.to_numeric(clean_df[col], errors='coerce')
            mu = x.mean()
            impute_mask = x.isna()
            clean_df.loc[impute_mask, col] = mu
            clean_df[col + "_imputed_flag"] = impute_mask.astype(int)
            print(f"[{col}] mean impute with {mu:.3f}")
        elif strat == 'mode':
            mode_val = clean_df[col].mode(dropna=True)
            mode_val = mode_val.iloc[0] if not mode_val.empty else None
            impute_mask = clean_df[col].isna()
            clean_df.loc[impute_mask, col] = mode_val
            clean_df[col + "_imputed_flag"] = impute_mask.astype(int)
            print(f"[{col}] mode impute with {mode_val}")
        elif strat.startswith('group_median:'):
            by_col = strat.split(':', 1)[1]
            if by_col not in clean_df.columns:
                x = pd.to_numeric(clean_df[col], errors='coerce')
                med = x.median()
                impute_mask = x.isna()
                clean_df.loc[impute_mask, col] = med
                clean_df[col + "_imputed_flag"] = impute_mask.astype(int)
                print(f"[{col}] fallback global median {med:.3f} (group col '{by_col}' missing)")
            else:
                x = pd.to_numeric(clean_df[col], errors='coerce')
                impute_mask = x.isna()
                group_meds = clean_df.groupby(by_col)[col].transform(lambda s: pd.to_numeric(s, errors='coerce').median())
                global_med = pd.to_numeric(clean_df[col], errors='coerce').median()
                fill_vals = group_meds.fillna(global_med)
                clean_df.loc[impute_mask, col] = fill_vals[impute_mask]
                clean_df[col + "_imputed_flag"] = impute_mask.astype(int)
                print(f"[{col}] group median by '{by_col}' (fallback to global median)")
        elif strat.startswith('constant:'):
            const_val = strat.split(':', 1)[1]
            try:
                const_val = float(const_val)
            except:
                pass
            impute_mask = clean_df[col].isna()
            clean_df.loc[impute_mask, col] = const_val
            clean_df[col + "_imputed_flag"] = impute_mask.astype(int)
            print(f"[{col}] constant impute with {const_val}")
        else:
            print(f"[{col}] Unknown strategy '{strat}'. Skipping.")

    display(clean_df[to_clean + [c + "_imputed_flag" for c in to_clean if c in clean_df.columns]].head(10))

# Part III: Reflection & Disclosure (20 points)
### Q5. (15 pts) Reflection
> **Process:** Loaded and inspected schema/dictionary; profiled missingness; identified key housing/demographic variables; detected outliers with IQR; selected imputation strategies (favoring group medians by state where appropriate); created imputation flags for transparency.  
> **Surprises:** _Fill after analysis (e.g., extreme vacancy pockets; valid but heavy-tailed price distributions)_.  
> **Limitations:** Cross-sectional limits causal claims; imputation choices may influence downstream modeling; documented choices and flags to mitigate.

---
## Appendix (Optional)
Add quick plots (histograms/boxplots) and per-state summaries to sanity-check distributions.